# Витрина этапов обучения `mart_student_stage`

Источник: `data/processed/mart_events.parquet`.

В одном Parquet храним два уровня:
- `aggregation_level='stage'` — одна строка = этап;
- `aggregation_level='stage_part'` — одна строка = этап × `part`.

Этапы:
`1–10`, `11–20`, `21–50`, `51–100`, `101–200`, `201–500`, `501+`.

Для вопроса stage определяется по `question_number`.
Для лекции — по `questions_before + 1`, то есть по текущей точке прогресса пользователя.

## 1. Настройки

In [1]:
# !pip install duckdb pandas pyarrow psutil

from pathlib import Path
import os
import time
import duckdb
import pandas as pd

if (Path("data") / "processed" / "mart_events.parquet").exists():
    DATA_DIR = Path("data")
elif (Path("processed") / "mart_events.parquet").exists():
    DATA_DIR = Path(".")
else:
    raise FileNotFoundError("Сначала построй data/processed/mart_events.parquet")

PROCESSED_DIR = DATA_DIR / "processed"
TEMP_DIR = DATA_DIR / "tmp_duckdb"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

MART_EVENTS_PATH = PROCESSED_DIR / "mart_events.parquet"
OUTPUT_PATH = PROCESSED_DIR / "mart_student_stage.parquet"

def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

MART_EVENTS = sql_path(MART_EVENTS_PATH)
OUTPUT = sql_path(OUTPUT_PATH)
TEMP = sql_path(TEMP_DIR)

MIN_STUDENTS_RELIABLE = 100
USE_APPROX_MEDIAN = True

threads = min(8, max(4, os.cpu_count() or 4))

try:
    import psutil
    total_ram_gb = psutil.virtual_memory().total / 1024**3
    memory_gb = max(2, min(8, int(total_ram_gb * 0.55)))
except Exception:
    memory_gb = 4

con = duckdb.connect()
con.execute(f"SET threads = {threads}")
con.execute(f"SET memory_limit = '{memory_gb}GB'")
con.execute(f"SET temp_directory = '{TEMP}'")
con.execute("SET preserve_insertion_order = false")

print("Источник:", MART_EVENTS_PATH.resolve())
print("Результат:", OUTPUT_PATH.resolve())
print("Threads:", threads)
print("Memory limit:", f"{memory_gb}GB")

Источник: D:\RACP\data\processed\mart_events.parquet
Результат: D:\RACP\data\processed\mart_student_stage.parquet
Threads: 8
Memory limit: 8GB


## 2. Проверяем схему `mart_events`

In [2]:
required = {
    "user_id", "content_type_id", "question_number", "questions_before",
    "answered_correctly", "prior_question_elapsed_time",
    "prior_question_had_explanation", "error_streak",
    "session_id", "part"
}

schema = con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{MART_EVENTS}')
""").df()

display(schema)

missing = required - set(schema["column_name"])
assert not missing, f"Не хватает полей: {sorted(missing)}"

print("✅ Схема подходит.")

,column_name,column_type,null,key,default,extra
0,row_id,BIGINT,YES,None,None,None
1,user_id,INTEGER,YES,None,None,None
2,timestamp,BIGINT,YES,None,None,None
3,content_id,INTEGER,YES,None,None,None
4,content_type_id,TINYINT,YES,None,None,None
5,content_kind,VARCHAR,YES,None,None,None
6,task_container_id,INTEGER,YES,None,None,None
7,user_answer,SMALLINT,YES,None,None,None
8,answered_correctly,SMALLINT,YES,None,None,None
9,prior_question_elapsed_time,DOUBLE,YES,None,None,None


✅ Схема подходит.


## 3. Проверяем границы этапов

Сначала смотрим распределение общего числа вопросов на пользователя и число пользователей,
которые достигают начала каждого этапа.

Границы не меняются автоматически: если `501+` окажется слишком маленьким,
ноутбук выведет предупреждение.

In [3]:
t0 = time.perf_counter()

distribution = con.sql(f"""
WITH uq AS (
    SELECT user_id, MAX(question_number) AS questions_total
    FROM read_parquet('{MART_EVENTS}')
    WHERE content_type_id = 0
    GROUP BY user_id
)
SELECT
    COUNT(*) AS students,
    AVG(questions_total) AS mean_questions,
    approx_quantile(questions_total, 0.50) AS p50,
    approx_quantile(questions_total, 0.75) AS p75,
    approx_quantile(questions_total, 0.90) AS p90,
    approx_quantile(questions_total, 0.95) AS p95,
    approx_quantile(questions_total, 0.99) AS p99,
    MAX(questions_total) AS max_questions
FROM uq
""").df()

display(distribution)

stage_reach = con.sql(f"""
WITH uq AS (
    SELECT user_id, MAX(question_number) AS questions_total
    FROM read_parquet('{MART_EVENTS}')
    WHERE content_type_id = 0
    GROUP BY user_id
)
SELECT * FROM (
    SELECT 1 stage_order, '1–10' stage,
           COUNT(*) FILTER (WHERE questions_total >= 1) students_reached FROM uq
    UNION ALL
    SELECT 2, '11–20',
           COUNT(*) FILTER (WHERE questions_total >= 11) FROM uq
    UNION ALL
    SELECT 3, '21–50',
           COUNT(*) FILTER (WHERE questions_total >= 21) FROM uq
    UNION ALL
    SELECT 4, '51–100',
           COUNT(*) FILTER (WHERE questions_total >= 51) FROM uq
    UNION ALL
    SELECT 5, '101–200',
           COUNT(*) FILTER (WHERE questions_total >= 101) FROM uq
    UNION ALL
    SELECT 6, '201–500',
           COUNT(*) FILTER (WHERE questions_total >= 201) FROM uq
    UNION ALL
    SELECT 7, '501+',
           COUNT(*) FILTER (WHERE questions_total >= 501) FROM uq
)
ORDER BY stage_order
""").df()

stage_reach["share_of_stage_1"] = (
    stage_reach["students_reached"] / stage_reach.loc[0, "students_reached"]
)

display(stage_reach)

late_n = int(stage_reach.loc[stage_reach.stage_order == 7, "students_reached"].iloc[0])
if late_n < MIN_STUDENTS_RELIABLE:
    print(f"⚠️ В 501+ только {late_n} пользователей: можно рассмотреть 201+.")
else:
    print(f"✅ В 501+ достаточно пользователей: {late_n:,}")

print(f"Проверка границ: {time.perf_counter() - t0:.1f} сек.")

,students,mean_questions,p50,p75,p90,p95,p99,max_questions
0,393656,252.17779,40.970629,154.78155,577.907012,1153.474583,3430.927737,17609.0


,stage_order,stage,students_reached,share_of_stage_1
0,1,1–10,393656,1.000000
1,2,11–20,390686,0.992455
2,3,21–50,332232,0.843965
3,4,51–100,174023,0.442069
4,5,101–200,123365,0.313383
5,6,201–500,84653,0.215043
6,7,501+,44264,0.112443


✅ В 501+ достаточно пользователей: 44,264
Проверка границ: 1.8 сек.


## 4. Присваиваем stage каждому событию

In [4]:
con.execute(f"""
CREATE OR REPLACE VIEW stage_events AS
WITH base AS (
    SELECT
        *,
        CASE
            WHEN content_type_id = 0 THEN CAST(question_number AS BIGINT)
            ELSE CAST(questions_before + 1 AS BIGINT)
        END AS progress_question_number
    FROM read_parquet('{MART_EVENTS}')
)
SELECT
    *,
    CASE
        WHEN progress_question_number BETWEEN 1 AND 10 THEN 1
        WHEN progress_question_number BETWEEN 11 AND 20 THEN 2
        WHEN progress_question_number BETWEEN 21 AND 50 THEN 3
        WHEN progress_question_number BETWEEN 51 AND 100 THEN 4
        WHEN progress_question_number BETWEEN 101 AND 200 THEN 5
        WHEN progress_question_number BETWEEN 201 AND 500 THEN 6
        WHEN progress_question_number >= 501 THEN 7
    END AS stage_order,
    CASE
        WHEN progress_question_number BETWEEN 1 AND 10 THEN '1–10'
        WHEN progress_question_number BETWEEN 11 AND 20 THEN '11–20'
        WHEN progress_question_number BETWEEN 21 AND 50 THEN '21–50'
        WHEN progress_question_number BETWEEN 51 AND 100 THEN '51–100'
        WHEN progress_question_number BETWEEN 101 AND 200 THEN '101–200'
        WHEN progress_question_number BETWEEN 201 AND 500 THEN '201–500'
        WHEN progress_question_number >= 501 THEN '501+'
    END AS stage,
    CASE
        WHEN progress_question_number BETWEEN 1 AND 10 THEN 1
        WHEN progress_question_number BETWEEN 11 AND 20 THEN 11
        WHEN progress_question_number BETWEEN 21 AND 50 THEN 21
        WHEN progress_question_number BETWEEN 51 AND 100 THEN 51
        WHEN progress_question_number BETWEEN 101 AND 200 THEN 101
        WHEN progress_question_number BETWEEN 201 AND 500 THEN 201
        WHEN progress_question_number >= 501 THEN 501
    END AS stage_min_question,
    CASE
        WHEN progress_question_number BETWEEN 1 AND 10 THEN 10
        WHEN progress_question_number BETWEEN 11 AND 20 THEN 20
        WHEN progress_question_number BETWEEN 21 AND 50 THEN 50
        WHEN progress_question_number BETWEEN 51 AND 100 THEN 100
        WHEN progress_question_number BETWEEN 101 AND 200 THEN 200
        WHEN progress_question_number BETWEEN 201 AND 500 THEN 500
        ELSE NULL
    END AS stage_max_question
FROM base
""")

display(con.sql("""
SELECT user_id, content_type_id, question_number, questions_before,
       progress_question_number, stage_order, stage, part
FROM stage_events
LIMIT 20
""").df())

,user_id,content_type_id,question_number,questions_before,progress_question_number,stage_order,stage,part
0,40828,0,1.0,0.0,1,1,1–10,1
1,40828,0,2.0,1.0,2,1,1–10,1
2,40828,0,3.0,2.0,3,1,1–10,1
3,40828,0,4.0,3.0,4,1,1–10,2
4,40828,0,5.0,4.0,5,1,1–10,3
5,40828,0,6.0,5.0,6,1,1–10,3
6,40828,0,7.0,6.0,7,1,1–10,3
7,40828,0,8.0,7.0,8,1,1–10,4
8,40828,0,9.0,8.0,9,1,1–10,4
9,40828,0,10.0,9.0,10,1,1–10,4


## 5. Основные метрики

`students_count` — уникальные пользователи с любым событием на этапе.

`lecture_rate` = пользователи с ≥1 лекцией / `students_count`.

`explanation_rate` — среднее `prior_question_had_explanation` по question events,
где поле не `NULL`.

`median_elapsed_time` строится из `prior_question_elapsed_time`.
По умолчанию используется `approx_quantile(..., 0.5)` для скорости на полном Riiid.

In [5]:
median_elapsed = (
    "approx_quantile(prior_question_elapsed_time, 0.5)"
    if USE_APPROX_MEDIAN else
    "median(prior_question_elapsed_time)"
)

median_streak = (
    "approx_quantile(error_streak, 0.5)"
    if USE_APPROX_MEDIAN else
    "median(error_streak)"
)

t0 = time.perf_counter()

event_metrics = con.sql(f"""
SELECT
    CASE WHEN GROUPING(part) = 1 THEN 'stage' ELSE 'stage_part' END
        AS aggregation_level,
    stage_order,
    stage,
    MIN(stage_min_question) AS stage_min_question,
    MAX(stage_max_question) AS stage_max_question,
    CASE WHEN GROUPING(part) = 1 THEN NULL ELSE part END AS part,

    COUNT(DISTINCT user_id) AS students_count,

    COUNT(*) FILTER (WHERE content_type_id = 0) AS answers_count,

    AVG(answered_correctly)
        FILTER (WHERE content_type_id = 0) AS accuracy,

    {median_elapsed}
        FILTER (
            WHERE content_type_id = 0
              AND prior_question_elapsed_time IS NOT NULL
        ) AS median_elapsed_time,

    AVG(prior_question_elapsed_time)
        FILTER (
            WHERE content_type_id = 0
              AND prior_question_elapsed_time IS NOT NULL
        ) AS mean_elapsed_time,

    COUNT(*) FILTER (
        WHERE content_type_id = 0
          AND prior_question_elapsed_time IS NOT NULL
    ) AS elapsed_time_observations,

    COUNT(*) FILTER (WHERE content_type_id = 1) AS lectures_count,

    COUNT(DISTINCT user_id)
        FILTER (WHERE content_type_id = 1) AS lecture_students_count,

    AVG(CAST(prior_question_had_explanation AS INTEGER))
        FILTER (
            WHERE content_type_id = 0
              AND prior_question_had_explanation IS NOT NULL
        ) AS explanation_rate,

    COUNT(*) FILTER (
        WHERE content_type_id = 0
          AND prior_question_had_explanation IS NOT NULL
    ) AS explanation_observations,

    {median_streak}
        FILTER (WHERE content_type_id = 0) AS median_error_streak

FROM stage_events
WHERE stage_order IS NOT NULL

GROUP BY GROUPING SETS (
    (stage_order, stage),
    (stage_order, stage, part)
)
""").df()

display(event_metrics.sort_values(
    ["aggregation_level", "stage_order", "part"],
    na_position="first"
).head(30))

print(f"Основные метрики: {time.perf_counter() - t0:.1f} сек.")

,aggregation_level,stage_order,stage,stage_min_question,stage_max_question,part,students_count,answers_count,accuracy,median_elapsed_time,mean_elapsed_time,elapsed_time_observations,lectures_count,lecture_students_count,explanation_rate,explanation_observations,median_error_streak
27,stage,1,1–10,1,10,<NA>,393656,3928411,0.507141,21223.362458,24322.908859,3535905,2159,2032,0.081842,3535905,0
36,stage,2,11–20,11,20,<NA>,390687,3672430,0.502691,19986.401395,22313.372812,3672430,19382,18128,0.516852,3672430,0
48,stage,3,21–50,21,50,<NA>,332257,7224103,0.585427,20019.458913,24285.175886,7224103,86141,65726,0.735129,7224103,0
28,stage,4,51–100,51,100,<NA>,174074,7253473,0.644936,19968.377130,24240.644575,7253473,144372,83470,0.942556,7253473,0
29,stage,5,101–200,101,200,<NA>,123379,10137455,0.654587,20478.837354,25256.813361,10137455,241606,81703,0.956895,10137455,0
14,stage,6,201–500,201,500,<NA>,84659,18144113,0.664490,21034.630120,26044.997359,18144113,482527,65614,0.977185,18144113,0
47,stage,7,501+,501,<NA>,<NA>,44266,48911315,0.691182,21179.477176,25884.751293,48911315,982845,36981,0.979732,48911315,0
5,stage_part,1,1–10,1,10,1,245788,723652,0.618609,21000.000000,22705.149703,557966,541,522,0.053396,557966,0
50,stage_part,1,1–10,1,10,2,295454,529097,0.613097,21000.051784,23471.823021,527120,700,680,0.199268,527120,1
25,stage_part,1,1–10,1,10,3,149583,448886,0.462449,17000.000000,18296.507057,448886,14,12,0.003048,448886,0


Основные метрики: 27.0 сек.


## 6. `avg_questions_per_session`

In [6]:
t0 = time.perf_counter()

session_metrics = con.sql("""
WITH session_level AS (
    SELECT
        CASE WHEN GROUPING(part) = 1 THEN 'stage' ELSE 'stage_part' END
            AS aggregation_level,
        stage_order,
        stage,
        CASE WHEN GROUPING(part) = 1 THEN NULL ELSE part END AS part,
        user_id,
        session_id,
        SUM(CASE WHEN content_type_id = 0 THEN 1 ELSE 0 END)
            AS questions_in_session
    FROM stage_events
    WHERE stage_order IS NOT NULL
    GROUP BY GROUPING SETS (
        (stage_order, stage, user_id, session_id),
        (stage_order, stage, part, user_id, session_id)
    )
)
SELECT
    aggregation_level,
    stage_order,
    stage,
    part,
    COUNT(*) AS sessions_count,
    AVG(questions_in_session) AS avg_questions_per_session
FROM session_level
GROUP BY aggregation_level, stage_order, stage, part
""").df()

display(session_metrics.sort_values(
    ["aggregation_level", "stage_order", "part"],
    na_position="first"
).head(30))

print(f"Сессионные метрики: {time.perf_counter() - t0:.1f} сек.")

,aggregation_level,stage_order,stage,part,sessions_count,avg_questions_per_session
44,stage,1,1–10,<NA>,475856,8.255462
33,stage,2,11–20,<NA>,482973,7.603800
43,stage,3,21–50,<NA>,682931,10.578086
25,stage,4,51–100,<NA>,633421,11.451267
32,stage,5,101–200,<NA>,780394,12.990175
24,stage,6,201–500,<NA>,1248021,14.538307
52,stage,7,501+,<NA>,2716513,18.005183
14,stage_part,1,1–10,1,256113,2.825518
20,stage_part,1,1–10,2,306069,1.728685
8,stage_part,1,1–10,3,149604,3.000495


Сессионные метрики: 15.7 сек.


## 7. Собираем финальную таблицу

In [7]:
mart_stage = event_metrics.merge(
    session_metrics,
    on=["aggregation_level", "stage_order", "stage", "part"],
    how="left",
    validate="one_to_one",
)

mart_stage["students_without_lectures"] = (
    mart_stage["students_count"] - mart_stage["lecture_students_count"]
)

mart_stage["lecture_rate"] = (
    mart_stage["lecture_students_count"] / mart_stage["students_count"]
)

mart_stage["lectures_per_student"] = (
    mart_stage["lectures_count"] / mart_stage["students_count"]
)

mart_stage["small_sample_flag"] = (
    mart_stage["students_count"] < MIN_STUDENTS_RELIABLE
)

mart_stage["sample_size_status"] = pd.cut(
    mart_stage["students_count"],
    bins=[-1, 99, 499, float("inf")],
    labels=["small", "medium", "ok"],
)

columns = [
    "aggregation_level", "stage_order", "stage",
    "stage_min_question", "stage_max_question", "part",
    "students_count", "answers_count", "accuracy",
    "median_elapsed_time", "mean_elapsed_time",
    "elapsed_time_observations",
    "lectures_count", "lecture_students_count",
    "students_without_lectures", "lecture_rate",
    "lectures_per_student",
    "explanation_rate", "explanation_observations",
    "median_error_streak",
    "sessions_count", "avg_questions_per_session",
    "small_sample_flag", "sample_size_status"
]

mart_stage = (
    mart_stage[columns]
    .sort_values(["aggregation_level", "stage_order", "part"],
                 na_position="first")
    .reset_index(drop=True)
)

display(mart_stage)

,aggregation_level,stage_order,stage,stage_min_question,stage_max_question,part,students_count,answers_count,accuracy,median_elapsed_time,...,students_without_lectures,lecture_rate,lectures_per_student,explanation_rate,explanation_observations,median_error_streak,sessions_count,avg_questions_per_session,small_sample_flag,sample_size_status
0,stage,1,1–10,1,10,<NA>,393656,3928411,0.507141,21223.362458,...,391624,0.005162,0.005484,0.081842,3535905,0,475856,8.255462,False,ok
1,stage,2,11–20,11,20,<NA>,390687,3672430,0.502691,19986.401395,...,372559,0.046400,0.049610,0.516852,3672430,0,482973,7.603800,False,ok
2,stage,3,21–50,21,50,<NA>,332257,7224103,0.585427,20019.458913,...,266531,0.197817,0.259260,0.735129,7224103,0,682931,10.578086,False,ok
3,stage,4,51–100,51,100,<NA>,174074,7253473,0.644936,19968.377130,...,90604,0.479509,0.829371,0.942556,7253473,0,633421,11.451267,False,ok
4,stage,5,101–200,101,200,<NA>,123379,10137455,0.654587,20478.837354,...,41676,0.662212,1.958242,0.956895,10137455,0,780394,12.990175,False,ok
5,stage,6,201–500,201,500,<NA>,84659,18144113,0.664490,21034.630120,...,19045,0.775039,5.699654,0.977185,18144113,0,1248021,14.538307,False,ok
6,stage,7,501+,501,<NA>,<NA>,44266,48911315,0.691182,21179.477176,...,7285,0.835427,22.203158,0.979732,48911315,0,2716513,18.005183,False,ok
7,stage_part,1,1–10,1,10,1,245788,723652,0.618609,21000.000000,...,245266,0.002124,0.002201,0.053396,557966,0,256113,2.825518,False,ok
8,stage_part,1,1–10,1,10,2,295454,529097,0.613097,21000.051784,...,294774,0.002302,0.002369,0.199268,527120,1,306069,1.728685,False,ok
9,stage_part,1,1–10,1,10,3,149583,448886,0.462449,17000.000000,...,149571,0.000080,0.000094,0.003048,448886,0,149604,3.000495,False,ok


## 8. Data Quality

In [8]:
# Каждый question event попал ровно в один stage.
coverage = con.sql("""
SELECT
    COUNT(*) FILTER (WHERE content_type_id = 0) AS questions_total,
    COUNT(*) FILTER (
        WHERE content_type_id = 0 AND stage_order IS NOT NULL
    ) AS questions_with_stage,
    COUNT(*) FILTER (
        WHERE content_type_id = 0 AND stage_order IS NULL
    ) AS questions_without_stage
FROM stage_events
""").df()

display(coverage)

assert int(coverage.loc[0, "questions_without_stage"]) == 0
assert int(coverage.loc[0, "questions_total"]) == int(
    coverage.loc[0, "questions_with_stage"]
)

# Stage question events соответствует пользовательскому question_number.
bad_stage = con.sql("""
SELECT COUNT(*)
FROM stage_events
WHERE content_type_id = 0
  AND (
       (stage_order = 1 AND NOT question_number BETWEEN 1 AND 10)
    OR (stage_order = 2 AND NOT question_number BETWEEN 11 AND 20)
    OR (stage_order = 3 AND NOT question_number BETWEEN 21 AND 50)
    OR (stage_order = 4 AND NOT question_number BETWEEN 51 AND 100)
    OR (stage_order = 5 AND NOT question_number BETWEEN 101 AND 200)
    OR (stage_order = 6 AND NOT question_number BETWEEN 201 AND 500)
    OR (stage_order = 7 AND question_number < 501)
  )
""").fetchone()[0]

assert bad_stage == 0

# Диапазоны.
for col in ["accuracy", "lecture_rate", "explanation_rate"]:
    vals = mart_stage[col].dropna()
    assert vals.between(0, 1).all(), f"{col} вне [0,1]"

# Общий stage: число пользователей не должно расти на поздних стадиях.
overall = (
    mart_stage[mart_stage["aggregation_level"] == "stage"]
    .sort_values("stage_order")
)

display(overall[
    ["stage_order", "stage", "students_count",
     "answers_count", "small_sample_flag"]
])

counts = overall["students_count"].tolist()
assert all(counts[i] >= counts[i + 1] for i in range(len(counts) - 1))

# Маленькие выборки отмечены.
assert (
    mart_stage["small_sample_flag"]
    == (mart_stage["students_count"] < MIN_STUDENTS_RELIABLE)
).all()

assert {"stage", "stage_part"} <= set(mart_stage["aggregation_level"])

print("✅ Все DQ-проверки пройдены.")

,questions_total,questions_with_stage,questions_without_stage
0,99271300,99271300,0


,stage_order,stage,students_count,answers_count,small_sample_flag
0,1,1–10,393656,3928411,False
1,2,11–20,390687,3672430,False
2,3,21–50,332257,7224103,False
3,4,51–100,174074,7253473,False
4,5,101–200,123379,10137455,False
5,6,201–500,84659,18144113,False
6,7,501+,44266,48911315,False


✅ Все DQ-проверки пройдены.


## 9. Проверяем готовность для графиков

In [9]:
print("Графики 1–4:")
display(
    mart_stage[
        mart_stage["aggregation_level"] == "stage"
    ][[
        "stage_order", "stage", "students_count",
        "accuracy", "median_elapsed_time",
        "lecture_rate", "explanation_rate",
        "small_sample_flag"
    ]]
)

print("График 5 — accuracy по stage × part:")
display(
    mart_stage[
        mart_stage["aggregation_level"] == "stage_part"
    ][[
        "stage_order", "stage", "part",
        "students_count", "accuracy", "small_sample_flag"
    ]].head(40)
)

Графики 1–4:


,stage_order,stage,students_count,accuracy,median_elapsed_time,lecture_rate,explanation_rate,small_sample_flag
0,1,1–10,393656,0.507141,21223.362458,0.005162,0.081842,False
1,2,11–20,390687,0.502691,19986.401395,0.046400,0.516852,False
2,3,21–50,332257,0.585427,20019.458913,0.197817,0.735129,False
3,4,51–100,174074,0.644936,19968.377130,0.479509,0.942556,False
4,5,101–200,123379,0.654587,20478.837354,0.662212,0.956895,False
5,6,201–500,84659,0.664490,21034.630120,0.775039,0.977185,False
6,7,501+,44266,0.691182,21179.477176,0.835427,0.979732,False


График 5 — accuracy по stage × part:


,stage_order,stage,part,students_count,accuracy,small_sample_flag
7,1,1–10,1,245788,0.618609,False
8,1,1–10,2,295454,0.613097,False
9,1,1–10,3,149583,0.462449,False
10,1,1–10,4,149358,0.364818,False
11,1,1–10,5,229216,0.476852,False
12,1,1–10,6,3448,0.579552,False
13,1,1–10,7,930,0.586280,False
14,2,11–20,1,45882,0.702744,False
15,2,11–20,2,116138,0.647976,False
16,2,11–20,3,4001,0.555453,False


## 10. Сохраняем Parquet

In [10]:
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

mart_stage.to_parquet(
    OUTPUT_PATH,
    index=False,
    compression="snappy"
)

print("✅ Сохранено:", OUTPUT_PATH.resolve())
print("Строк:", len(mart_stage))
print(f"Размер: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

✅ Сохранено: D:\RACP\data\processed\mart_student_stage.parquet
Строк: 56
Размер: 21.5 KB


## 11. Финальная проверка файла

In [11]:
saved = pd.read_parquet(OUTPUT_PATH)

assert len(saved) == len(mart_stage)
assert saved["accuracy"].dropna().between(0, 1).all()
assert saved["lecture_rate"].dropna().between(0, 1).all()
assert saved["explanation_rate"].dropna().between(0, 1).all()

display(saved.head(20))
print("✅ mart_student_stage.parquet готова.")

,aggregation_level,stage_order,stage,stage_min_question,stage_max_question,part,students_count,answers_count,accuracy,median_elapsed_time,...,students_without_lectures,lecture_rate,lectures_per_student,explanation_rate,explanation_observations,median_error_streak,sessions_count,avg_questions_per_session,small_sample_flag,sample_size_status
0,stage,1,1–10,1,10,<NA>,393656,3928411,0.507141,21223.362458,...,391624,0.005162,0.005484,0.081842,3535905,0,475856,8.255462,False,ok
1,stage,2,11–20,11,20,<NA>,390687,3672430,0.502691,19986.401395,...,372559,0.046400,0.049610,0.516852,3672430,0,482973,7.603800,False,ok
2,stage,3,21–50,21,50,<NA>,332257,7224103,0.585427,20019.458913,...,266531,0.197817,0.259260,0.735129,7224103,0,682931,10.578086,False,ok
3,stage,4,51–100,51,100,<NA>,174074,7253473,0.644936,19968.377130,...,90604,0.479509,0.829371,0.942556,7253473,0,633421,11.451267,False,ok
4,stage,5,101–200,101,200,<NA>,123379,10137455,0.654587,20478.837354,...,41676,0.662212,1.958242,0.956895,10137455,0,780394,12.990175,False,ok
5,stage,6,201–500,201,500,<NA>,84659,18144113,0.664490,21034.630120,...,19045,0.775039,5.699654,0.977185,18144113,0,1248021,14.538307,False,ok
6,stage,7,501+,501,<NA>,<NA>,44266,48911315,0.691182,21179.477176,...,7285,0.835427,22.203158,0.979732,48911315,0,2716513,18.005183,False,ok
7,stage_part,1,1–10,1,10,1,245788,723652,0.618609,21000.000000,...,245266,0.002124,0.002201,0.053396,557966,0,256113,2.825518,False,ok
8,stage_part,1,1–10,1,10,2,295454,529097,0.613097,21000.051784,...,294774,0.002302,0.002369,0.199268,527120,1,306069,1.728685,False,ok
9,stage_part,1,1–10,1,10,3,149583,448886,0.462449,17000.000000,...,149571,0.000080,0.000094,0.003048,448886,0,149604,3.000495,False,ok


✅ mart_student_stage.parquet готова.


# Логика витрины

- Источник — только `mart_events`.
- Question stage определяется по `question_number`.
- Lecture stage определяется по `questions_before + 1`.
- `stage_order` нужен для корректной сортировки в BI.
- `lecture_rate` — доля пользователей этапа, у которых была хотя бы одна лекция.
- `explanation_rate` — доля True среди ненулевых `prior_question_had_explanation`.
- `avg_questions_per_session` считается через промежуточный уровень `stage × user × session`.
- `small_sample_flag=True`, если `students_count < 100`.
- Для скорости медианы по умолчанию приблизительные (`approx_quantile`).
- `stage` уже является progress segment, поэтому дополнительные комбинации
  `stage × part × lecture_segment × session_number` в первую версию не добавляем,
  чтобы не создавать маленькие группы.

Рост accuracy нельзя автоматически считать эффектом обучения:
на него также могут влиять сложность вопросов, состав продолжающих пользователей
и распределение тем/`part`.